# 06 — Adaptive RAG: Pre-Retrieval Routing

**Track:** Advanced · **Stage:** Production Patterns

In standard RAG pipelines, every query goes through the exact same flow: Embed Query -> Search VectorDB -> Generate Answer. 

This is inefficient. If a user says "Hello!", we waste time and money doing a vector search. If a user asks a complex multi-hop question, standard vector search will fail, but we don't realize it until it's too late.

**Adaptive RAG** solves this by evaluating the query *before* retrieval. An LLM acts as a Router, classifying the intent and complexity of the query, and directing it to the most appropriate sub-system.

In this comprehensive deep dive, we will:
1. **Part 1: The Theory of Query Classification.** Build a simple LLM router to classify queries.
2. **Part 2: Production Implementation.** Use **LangGraph** to build an Adaptive RAG state machine that conditionally routes queries to different specialized tools.

---
## Part 1: The Theory of Query Classification

Before we build the full graph, let's look at the core of Adaptive RAG: The Router Prompt. The router needs to output a structured decision.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.llms.fake import FakeListLLM
import json

class RouteDecision(BaseModel):
    datasource: str = Field(description="The datasource to use. Options: 'web_search', 'internal_search', 'direct_answer'")
    reasoning: str = Field(description="Why this datasource was chosen.")

# In production, you would use ChatOpenAI(model="gpt-4o-mini").with_structured_output(RouteDecision)
# Here we simulate structured output with a FakeListLLM
mock_router_llm = FakeListLLM(responses=[
    json.dumps({"datasource": "direct_answer", "reasoning": "The user is just saying hello."}),
    json.dumps({"datasource": "web_search", "reasoning": "The user is asking about current weather which requires web search."}),
    json.dumps({"datasource": "internal_search", "reasoning": "The user is asking about company policy."})
])

def route_query(query: str) -> dict:
    print(f"\nQuestion: {query}")
    response = mock_router_llm.invoke(f"Classify this query: {query}")
    decision = json.loads(response)
    print(f"  -> Route: {decision['datasource']}")
    print(f"  -> Reason: {decision['reasoning']}")
    return decision

print("--- Testing the Router ---")
route_query("Hi there!")
route_query("What is the weather in London today?")
route_query("What is our Q3 expense policy?")

---
## Part 2: Production Implementation with LangGraph

Now we'll tie this router into a `StateGraph`. The graph will have one entry point (the router) and three possible execution paths.

In [ ]:
# !pip install langgraph langchain langchain-core

from typing import List, Dict, TypedDict
from langgraph.graph import StateGraph, END
import random

### Step A: Define the State and Nodes

In [ ]:
class GraphState(TypedDict):
    question: str
    generation: str
    route: str

def router_node(state: GraphState):
    print("---NODE: ROUTER---")
    question = state["question"]
    # We'll use a simple heuristic to simulate the LLM Router for this tutorial
    if "hello" in question.lower():
        route = "direct_answer"
    elif "weather" in question.lower():
        route = "web_search"
    else:
        route = "internal_search"
    print(f"  [Router] Chose path: {route}")
    return {"route": route}

def direct_answer_node(state: GraphState):
    print("---NODE: DIRECT ANSWER (No Retrieval)---")
    return {"generation": "Hello! How can I help you today?"}

def web_search_node(state: GraphState):
    print("---NODE: WEB SEARCH---")
    # Simulate web search
    return {"generation": "Based on the web, it is sunny in London today."}

def internal_search_node(state: GraphState):
    print("---NODE: INTERNAL SEARCH---")
    # Simulate vector DB search
    return {"generation": "Based on company policy, Q3 expenses must be submitted by Friday."}

### Step B: Define Conditional Edges

The conditional edge looks at the state updated by the router node and determines the next node.

In [ ]:
def decide_route(state: GraphState):
    print("---EDGE: CONDITIONAL ROUTING---")
    return state["route"] # returns 'direct_answer', 'web_search', or 'internal_search'

### Step C: Compile and Run

We tie the nodes and edges together.

In [ ]:
workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("router", router_node)
workflow.add_node("direct_answer", direct_answer_node)
workflow.add_node("web_search", web_search_node)
workflow.add_node("internal_search", internal_search_node)

# Set entry point to the router
workflow.set_entry_point("router")

# The router determines which node executes next
workflow.add_conditional_edges(
    "router",
    decide_route,
    {
        "direct_answer": "direct_answer",
        "web_search": "web_search",
        "internal_search": "internal_search",
    }
)

# All paths lead to END
workflow.add_edge("direct_answer", END)
workflow.add_edge("web_search", END)
workflow.add_edge("internal_search", END)

app = workflow.compile()

def run_adaptive_rag(question: str):
    print(f"\n========== QUERY: '{question}' ==========")
    inputs = {"question": question}
    for output in app.stream(inputs):
        pass
    
    # Get the final generation from whichever node ran last
    final_node = list(output.keys())[0]
    print(f"\nFINAL ANSWER ({final_node}): {output[final_node]['generation']}\n")

# Test 1: Simple Greeting (Bypasses Retrieval)
run_adaptive_rag("Hello there!")

# Test 2: Current Events (Web Search)
run_adaptive_rag("What is the weather like?")

# Test 3: Internal Knowledge (Vector Search)
run_adaptive_rag("What is the policy for Q3?")

## Reflection

1. **Cost Savings:** By routing "Hello" directly to a static generation or a small model, we save the cost and latency of embedding the query and querying a vector database.
2. **Adaptive vs Corrective (CRAG):** These patterns are often combined! Adaptive RAG sits at the *front* (Pre-Retrieval Routing) to choose the right pipeline. Corrective RAG sits in the *middle* (Post-Retrieval Grading) to ensure the chosen pipeline actually succeeded.

## References
- [Adaptive-RAG: Learning to Adapt Retrieval-Augmented Large Language Models through Question Complexity](https://arxiv.org/pdf/2403.14403)